# Step 4 — Metrics Tracker

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Build a `MetricsTracker` and a `timed_ram_block` context manager to record performance and compute metrics (AUC-ROC, F1, ΔAU, retraining time, peak RAM) for each experiment run.

Reference implementation: `drift_framework/metrics/tracker.py`

## 4.1 Setup

In [ ]:
import sys, os, time, tracemalloc
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Generator, List
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score

SEED = 42
RESULTS_DIR = Path(PROJECT_ROOT) / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Setup OK")

## 4.2 ExperimentRecord — one row of results

In [ ]:
@dataclass
class ExperimentRecord:
    """One row of experiment results (matches CSV columns)."""
    dataset: str
    model: str
    drift_type: str
    severity: str
    start_idx: int
    auc_before: float
    auc_after: float
    delta_auc: float
    f1_before: float
    f1_after: float
    retrain_time_sec: float
    ram_mb: float

## 4.3 `timed_ram_block` — context manager for time and memory

In [ ]:
@contextmanager
def timed_ram_block() -> Generator[dict, None, None]:
    """
    Context manager that measures wall-clock time and peak RAM usage.

    Yields a dict that is populated on exit with:
      - "elapsed_sec" : float seconds elapsed
      - "peak_ram_mb" : float peak memory in MB (via tracemalloc)

    Usage:
        with timed_ram_block() as m:
            model.fit(X, y)
        print(m["elapsed_sec"], m["peak_ram_mb"])
    """
    result: dict = {}
    tracemalloc.start()
    t0 = time.perf_counter()
    try:
        yield result
    finally:
        elapsed = time.perf_counter() - t0
        _current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        result["elapsed_sec"] = elapsed
        result["peak_ram_mb"] = peak / (1024 ** 2)

In [ ]:
# Quick demo of timed_ram_block
with timed_ram_block() as m:
    # Simulate some work (allocate a large array)
    _ = np.random.default_rng(42).random((10_000, 100))
    time.sleep(0.05)

print(f"Elapsed : {m['elapsed_sec']:.3f} s")
print(f"Peak RAM: {m['peak_ram_mb']:.2f} MB")

## 4.4 MetricsTracker class

In [ ]:
class MetricsTracker:
    """Records per-run metrics across multiple experiment configurations."""

    def __init__(self):
        self._records: List[ExperimentRecord] = []

    def record(
        self,
        dataset: str,
        model: str,
        drift_type: str,
        severity: str,
        start_idx: int,
        y_true_before: pd.Series,
        y_prob_before: np.ndarray,
        y_true_after: pd.Series,
        y_prob_after: np.ndarray,
        retrain_time_sec: float,
        ram_mb: float,
    ) -> ExperimentRecord:
        """Compute and store AUC-ROC, F1, ΔAU, time, and RAM for one run."""
        auc_before = roc_auc_score(y_true_before, y_prob_before)
        auc_after  = roc_auc_score(y_true_after,  y_prob_after)
        delta_auc  = auc_before - auc_after

        f1_before = f1_score(y_true_before, (y_prob_before >= 0.5).astype(int))
        f1_after  = f1_score(y_true_after,  (y_prob_after  >= 0.5).astype(int))

        rec = ExperimentRecord(
            dataset=dataset, model=model, drift_type=drift_type,
            severity=severity, start_idx=start_idx,
            auc_before=auc_before, auc_after=auc_after, delta_auc=delta_auc,
            f1_before=f1_before, f1_after=f1_after,
            retrain_time_sec=retrain_time_sec, ram_mb=ram_mb,
        )
        self._records.append(rec)
        print(
            f"[MetricsTracker] {dataset}/{model}/{drift_type}/{severity}: "
            f"AUC {auc_before:.4f} → {auc_after:.4f}  (Δ={delta_auc:+.4f})  "
            f"time={retrain_time_sec:.2f}s  RAM={ram_mb:.1f}MB"
        )
        return rec

    def to_dataframe(self) -> pd.DataFrame:
        """Return all recorded experiments as a DataFrame."""
        return pd.DataFrame([vars(r) for r in self._records])

    def to_csv(self, path: str) -> None:
        """Write all recorded experiments to a CSV file."""
        df = self.to_dataframe()
        df.to_csv(path, index=False)
        print(f"[MetricsTracker] Saved {len(df)} records to {path}")

## 4.5 End-to-end example: baseline vs. drifted run

In [ ]:
from drift_framework.data.loader import load_dataset
from drift_framework.pipeline.two_stage import TwoStagePipeline
from drift_framework.drift.injector import DriftInjector

bundle = load_dataset("adult")

In [ ]:
# Train baseline pipeline and time it
pipe = TwoStagePipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)

with timed_ram_block() as train_m:
    pipe.fit(bundle.X_ref, bundle.y_ref)

print(f"Training time : {train_m['elapsed_sec']:.2f}s")
print(f"Peak RAM      : {train_m['peak_ram_mb']:.1f} MB")

In [ ]:
# Predict on clean pre-drift split (baseline performance)
proba_pre = pipe.predict_proba(bundle.X_pre)[:, 1]
print(f"Baseline AUC (pre-drift): {roc_auc_score(bundle.y_pre, proba_pre):.4f}")

In [ ]:
# Inject high covariate shift into the post-drift split
injector = DriftInjector(drift_type="covariate", severity="high")
injector.fit(bundle.X_ref, bundle.num_features)
X_post_drifted, y_post_drifted = injector.inject(bundle.X_post, bundle.y_post)

# Predict on drifted data (no retraining)
proba_post = pipe.predict_proba(X_post_drifted)[:, 1]
print(f"Post-drift AUC (no retraining): {roc_auc_score(y_post_drifted, proba_post):.4f}")

In [ ]:
# Record the full experiment
tracker = MetricsTracker()
tracker.record(
    dataset="adult",
    model="xgboost",
    drift_type="covariate",
    severity="high",
    start_idx=0,
    y_true_before=bundle.y_pre,
    y_prob_before=proba_pre,
    y_true_after=y_post_drifted,
    y_prob_after=proba_post,
    retrain_time_sec=train_m["elapsed_sec"],
    ram_mb=train_m["peak_ram_mb"],
)

## 4.6 Record a second run (concept drift) for comparison

In [ ]:
inj2 = DriftInjector(drift_type="concept", severity="medium")
inj2.fit(bundle.X_ref, bundle.num_features)
X_con, y_con = inj2.inject(bundle.X_post, bundle.y_post)
proba_con = pipe.predict_proba(X_con)[:, 1]

tracker.record(
    dataset="adult",
    model="xgboost",
    drift_type="concept",
    severity="medium",
    start_idx=0,
    y_true_before=bundle.y_pre,
    y_prob_before=proba_pre,
    y_true_after=y_con,
    y_prob_after=proba_con,
    retrain_time_sec=0.0,
    ram_mb=0.0,
)

## 4.7 Inspect and export results

In [ ]:
df = tracker.to_dataframe()
df

In [ ]:
out_path = str(RESULTS_DIR / "notebook_metrics.csv")
tracker.to_csv(out_path)

## 4.8 Sanity checks

In [ ]:
df = tracker.to_dataframe()

# 1. delta_auc = auc_before - auc_after
for _, row in df.iterrows():
    expected = row["auc_before"] - row["auc_after"]
    assert abs(row["delta_auc"] - expected) < 1e-9, \
        f"delta_auc mismatch: {row['delta_auc']} vs {expected}"
print("delta_auc = auc_before - auc_after — OK")

# 2. AUC values are in [0, 1]
for col in ["auc_before", "auc_after"]:
    assert df[col].between(0, 1).all(), f"{col} out of [0,1] range"
print("AUC values in [0, 1] — OK")

# 3. CSV file exists and has correct columns
loaded = pd.read_csv(out_path)
expected_cols = ["dataset", "model", "drift_type", "severity", "start_idx",
                 "auc_before", "auc_after", "delta_auc", "f1_before", "f1_after",
                 "retrain_time_sec", "ram_mb"]
for col in expected_cols:
    assert col in loaded.columns, f"Missing column '{col}' in CSV"
print("CSV has all expected columns — OK")

# 4. Framework tracker gives same results
from drift_framework.metrics.tracker import MetricsTracker as FWTracker
fw_tracker = FWTracker()
fw_tracker.record(
    dataset="adult", model="xgboost", drift_type="covariate", severity="high", start_idx=0,
    y_true_before=bundle.y_pre, y_prob_before=proba_pre,
    y_true_after=y_post_drifted, y_prob_after=proba_post,
    retrain_time_sec=train_m["elapsed_sec"], ram_mb=train_m["peak_ram_mb"],
)
fw_df = fw_tracker.to_dataframe()
nb_row = df[df["drift_type"] == "covariate"].iloc[0]
assert abs(fw_df.iloc[0]["delta_auc"] - nb_row["delta_auc"]) < 1e-9
print("Notebook tracker matches framework tracker — OK")

print("\nAll sanity checks passed!")